In [21]:
%pip install optuna nnaudio


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [22]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("You are not in a Google Colab environment. Saving locally")

You are not in a Google Colab environment. Saving locally


In [23]:
import sys
sys.path.insert(0,"/content/drive/MyDrive/CNN Training")

import importlib
import model
import dataset_loader
import algorithm

importlib.reload(algorithm)


<module 'algorithm' from 'c:\\Users\\Aljon Catiban\\Documents\\Bachelor of Science in Computer Engineering\\Thesis\\Realtime-Bass-Tablature-Transcription\\algorithm.py'>

In [24]:
import torch
import torch.optim as optim
import torch.nn as nn
import time
import numpy as np
import optuna
import csv
import os
import json

from torch.utils.data import DataLoader, random_split
from algorithm import AdaptiveMultiStageStressTestedHPO
from model import BassTranscriptionCNN
from dataset_loader import Dataset

In [25]:
print(torch.cuda.is_available())

True


In [26]:
def evaluate_model(config, train_loader, val_loader, stress_test, profile_latency):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = BassTranscriptionCNN(config).to(device)

    criterion_class = nn.CrossEntropyLoss()
    criterion_binary = nn.BCELoss()

    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])

    max_epochs = 15
    patience = 5
    best_val_loss = float('inf')
    patience_counter = 0

    if train_loader is not None and val_loader is not None:
        for epoch in range(max_epochs):
            model.train()
            for batch_audio, labels in train_loader:
                batch_audio= batch_audio.to(device)
                labels = {k: v.to(device) for k, v in labels.items()}

                out_string, out_fret, out_pitch, out_onset, out_offset = model(batch_audio)

                loss_string = criterion_class(out_string, labels['string'])
                loss_fret = criterion_class(out_fret, labels['fret'])
                loss_pitch = criterion_class(out_pitch, labels['pitch'].long())
                loss_onset = criterion_binary(out_onset, labels['onset'])
                loss_offset = criterion_binary(out_offset, labels['offset'])

                total_loss = loss_string + loss_fret + loss_pitch + loss_onset + loss_offset

                optimizer.zero_grad()
                total_loss.backward()
                optimizer.step()

            model.eval()
            val_loss_accumulated = 0.0
            with torch.no_grad():
                for val_audio, val_labels in val_loader:
                    val_audio = val_audio.to(device)
                    val_labels = {k: v.to(device) for k, v in val_labels.items()}

                    val_out_string, val_out_fret, val_out_pitch, val_out_onset, val_out_offset =  model(val_audio)

                    val_loss_string = criterion_class(val_out_string, val_labels['string'])
                    val_loss_fret = criterion_class(val_out_fret, val_labels['fret'])
                    val_loss_pitch = criterion_class(val_out_pitch, val_labels['pitch'])
                    val_loss_onset = criterion_binary(val_out_onset, val_labels['onset'])
                    val_loss_offset = criterion_binary(val_out_offset, val_labels['offset'])

                    val_loss_accumulated += (val_loss_string + val_loss_fret + val_loss_pitch + val_loss_onset + val_loss_offset).item()

            epoch_val_loss = val_loss_accumulated / len(val_loader)

            if epoch_val_loss < best_val_loss:
                best_val_loss = epoch_val_loss
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= patience:
                print(f"Early Stopping triggerd at Epoch {epoch + 1}. No improvement since last {patience} epochs.")
                break

    validation_loss = best_val_loss if best_val_loss != float('inf') else 1.0
    avg_latency = 0.0

    if profile_latency:
        model.eval()
        dummy_input = torch.randn(1, 4608).to(device)

        latencies = []

        with torch.no_grad():
            for _ in range(10):
                if stress_test:
                    noise_std = 0.05
                    network_input = dummy_input +  torch.randn_like(dummy_input) * noise_std
                else:
                    network_input = dummy_input

                t_input = time.perf_counter()
                _ = model(network_input)
                t_output = time.perf_counter()

                latencies.append((t_output - t_input) * 1000)
        avg_latency = sum(latencies) / len(latencies)

    return validation_loss, avg_latency


In [27]:
def create_optuna_objective(train_loader, val_loader):
    def optuna_objective(trial):
        try:
          config = {
              'learning_rate': trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True),
              'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.5),
              'activation_function': trial.suggest_categorical('activation_function', ['ReLU', 'Tanh', 'ELU']),
              'convolution_layers': trial.suggest_int('convolution_layers', 1,4),
              'filter_layers': trial.suggest_categorical('filter_layers', [16, 32, 64, 128]),
              'filter_size': trial.suggest_categorical('filter_size', [2, 3, 5, 7])
          }

          loss, latency = evaluate_model(config, train_loader, val_loader, stress_test=False, profile_latency=False)

          if loss is None:
              return -1.0

          return 1/(1.0 + loss)

        except Exception as e:
          print(f"Error at trial: {trial.number}", e)
          return -1.0

    return optuna_objective

In [28]:
def safe_eval(config, stress_test = False):
    try:
        return evaluate_model(config, train_loader, val_loader, stress_test, profile_latency=True)
    except Exception as e:
      print("Evaluaiton Failed: ", e)
      return 1.0, 9999

In [29]:
def optuna_csv_logger(study, trial, filename, mode):
    file_exist = os.path.isfile(filename)
    value = trial.value if trial.value is not None else -1.0

    print(f"[{mode}] Trial: {trial.number + 1} | Fitness: {trial.value:.4f}")

    with open(filename, mode='a', newline='') as f:
        writer = csv.writer(f)

        if not file_exist:
            writer.writerow(['Trial',
                             'Learning Rate',
                             'Dropout Rate',
                             'Activation Function',
                             'Convolution Layers',
                             'Filter Layers',
                             'Kernel Size',
                             'Fitness Score'])
        params = trial.params

        writer.writerow([trial.number + 1,
                        params.get('learning_rate', None),
                        params.get('dropout_rate', None),
                        params.get('activation_function', None),
                        params.get('convolution_layers', None),
                        params.get('filter_layers', None),
                        params.get('filter_size', None),
                        value])

In [ ]:
if __name__ == "__main__":

    print("Optimization Experiment")

    try:
        drive.mount('/content/drive')
        GDRIVE_PATH = "/content/drive/MyDrive/CNN Training"
    except:
        print("running locally")
        GDRIVE_PATH = r"c:\Users\Aljon Catiban\Documents\Bachelor of Science in Computer Engineering\Thesis\Realtime-Bass-Tablature-Transcription"


    if not os.path.exists(GDRIVE_PATH):
        os.makedirs(GDRIVE_PATH)
        print(f"Creating new folder at {GDRIVE_PATH}")
    else:
        print(f"Already linked in existing folder at {GDRIVE_PATH}") 



    MODE = "PROPOSED" #   PROPOSED, BAYESIAN OPTIMIZATION, RANDOM SEARCH
    TOTAL_TRIALS = 30

    print("Loading dataset")
    full_dataset = Dataset(csv_file = f"{GDRIVE_PATH}/databases/augmented_dataset_labels.csv",
                           root_dir = f"{GDRIVE_PATH}")

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    print(f"Training Samples: {len(train_dataset)} \nValidation Samples: {len(val_dataset)}")

    print(f"Starting optimization using {MODE} ALGORITHM")
    if MODE == "PROPOSED":
        logfile = os.path.join(GDRIVE_PATH, "hpo_proposed_log.csv")
        checkpoint = os.path.join(GDRIVE_PATH, "hpo_proposed_checkpoint.pkl")

        hpo_engine = AdaptiveMultiStageStressTestedHPO(budgetTrial=30,
                                                       alpha=0.5,
                                                       beta=0.5,
                                                       rho_S=0.6,
                                                       rho_R=0.4,
                                                       gamma=0.3,
                                                       logfile=logfile,
                                                       checkpoint=checkpoint)
        train_eval_wrapper = safe_eval
        best_config = hpo_engine.optimization(train_eval_wrapper)

    elif MODE in ["BAYESIAN OPTIMIZATION", "RANDOM SEARCH"]:
        database_path = f"sqlite:///{os.path.join(GDRIVE_PATH, f'optuna_{MODE.lower()}_study.db')}"
        logfile = os.path.join(GDRIVE_PATH, f"optuna_{MODE.lower()}_log.csv")

        sampler = optuna.samplers.RandomSampler() if MODE == "RANDOM SEARCH" else optuna.samplers.TPESampler()
        study = optuna.create_study(direction='maximize',
                                    sampler=sampler,
                                    study_name=f"optimization_{MODE.lower()}",
                                    load_if_exists=True,
                                    storage=database_path)

        objective_function = create_optuna_objective(train_loader, val_loader)

        trials_left = TOTAL_TRIALS - len(study.trials)
        if trials_left > 0:
            print(f"Resuming optimization... {trials_left} trials left")
            study.optimize(objective_function,
                        n_trials=trials_left,
                        callbacks=[lambda s, t: optuna_csv_logger(s, t, logfile, MODE)])
        else:
            print("Optimization completed.")

        best_config = study.best_params

    print(f"Final retraining with best config : {best_config}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    final_model = BassTranscriptionCNN(best_config).to(device)

    criterion_class = nn.CrossEntropyLoss()
    criterion_binary = nn.BCELoss()

    optimizer = optim.Adam(final_model.parameters(), lr=best_config['learning_rate'])

    final_epochs = 50

    for epoch in range(final_epochs):
        final_model.train()
        total_epoch_loss = 0.0

        for batch_audio, labels, in train_loader:
            batch_audio = batch_audio.to(device)
            labels = {k: v.to(device) for k, v in labels.items()}

            optimizer.zero_grad()
            out_string, out_fret, out_pitch, out_onset, out_offset = final_model(batch_audio)

            loss_string = criterion_class(out_string, labels['string'])
            loss_fret = criterion_class(out_fret, labels['fret'])
            loss_pitch = criterion_class(out_pitch, labels['pitch'])
            loss_onset = criterion_binary(out_onset, labels['onset'])
            loss_offset = criterion_binary(out_offset, labels['offset'])

            total_loss = loss_string + loss_fret + loss_pitch + loss_onset + loss_offset

            total_loss.backward()
            optimizer.step()

            total_epoch_loss += total_loss.item()

        avg_loss = total_epoch_loss / len(train_loader)
        print(f"Final retraining - Epoch [{epoch + 1 } / {final_epochs}], Average Loss: {avg_loss: .4f}")

    config_save_path = os.path.join(GDRIVE_PATH, f"best_config_{MODE.lower()}.json")
    with open(config_save_path, 'w') as f:
        json.dump(best_config, f, indent=4)

    model_save_path = os.path.join(GDRIVE_PATH, f"trained_{MODE.lower()}.pth")
    torch.save(final_model.state_dict(), model_save_path)

    print(f"Trained weights saved as {model_save_path}")

Optimization Experiment
running locally
Already linked in existing folder at c:\Users\Aljon Catiban\Documents\Bachelor of Science in Computer Engineering\Thesis\Realtime-Bass-Tablature-Transcription
Loading dataset
Training Samples: 3655 
Validation Samples: 914
Starting optimization using PROPOSED ALGORITHM
--- HPO Budgets Initialized ---
Total Budget (B): 30 trials
Selection Budget (S): 18 | Refinement Budget (R): 12
Population Size (P): 6 | Generations (G): 3
-------------------------------
Entering selection stage

--- Generation 1/3 ---

[PROPOSED] Trial 1/30
Config → LR=0.000383 | Conv=2 | Filters=128 | Kernel=7
CQT kernels created, time used = 0.4267 seconds
Result → Loss=7.0987 | Latency=5.33ms | Fitness=0.2461

ABOUT TO WRITE CSV
CSV Logged
[PROPOSED] Trial 2/30
Config → LR=0.009380 | Conv=4 | Filters=32 | Kernel=5
CQT kernels created, time used = 0.3145 seconds
Early Stopping triggerd at Epoch 14. No improvement since last 5 epochs.
Result → Loss=7.1964 | Latency=5.63ms | Fit